# Load the Existing Vector Store

In [1]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Must use the same model used during ingestion
embedder = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedder,
    collection_name="company-docs"
)

print(f"Chunks in store: {vectorstore._collection.count()}")

/tmp/ipykernel_39769/1088421288.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/tmp/ipykernel_39769/1088421288.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(
/home/ml/miniconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook im

Chunks in store: 9


# Basic Similarity Search

In [2]:
query = "Was war das Hauptziel des Pilotversuchs?"

results = vectorstore.similarity_search(query, k=4)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(f"Text: {doc.page_content[:200]}...")


--- Result 1 ---
Source: chunking.pdf | Page: 4
Text: Testfall fuer Retrieval
Frage: "Hat die Barcode-Pruefung das Ziel allein erreicht?" Eine oberflaechliche Suche
koennte wegen der Tabellenzahl 2,1 Prozent mit "ja" antworten. Korrekt ist: Die Tabelle n...

--- Result 2 ---
Source: chunking.pdf | Page: 1
Text: RAG Chunking Demo
2
1. Ausgangslage und Ziel
Im ersten Quartal lag die Fehlkommissionierungsquote im Lager Nord bei 3,8 Prozent. Besonders
haeufig wurden aehnlich verpackte Artikel verwechselt. Das Pi...

--- Result 3 ---
Source: chunking.pdf | Page: 2
Text: Gesamtprogramm unterscheiden.
Einschraenkung: Die Aussage, dass die Barcode-Pruefung den groessten Effekt hatte, gilt nur
fuer dieses Lager und diesen achtwoechigen Zeitraum. Eine Uebertragung auf and...

--- Result 4 ---
Source: chunking.pdf | Page: 1
Text: Die Analyse identifizierte drei Hauptursachen: uneindeutige Stellplatzkennzeichnung, zu spaete
Plausibilitaetspruefung und unzureichende Rueckmeldung an neue Mitarbeite

#  Similarity Search with Scores

In [3]:
results_with_scores = vectorstore.similarity_search_with_score(query, k=4)

# Collection uses cosine space: score is cosine distance (1 - similarity), lower = more similar
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n--- Result {i+1} ---")
    print(f"Score: {score:.4f} (similarity: {1 - score:.4f})")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(f"Text: {doc.page_content[:200]}...")


--- Result 1 ---
Score: 0.5215 (similarity: 0.4785)
Source: chunking.pdf | Page: 4
Text: Testfall fuer Retrieval
Frage: "Hat die Barcode-Pruefung das Ziel allein erreicht?" Eine oberflaechliche Suche
koennte wegen der Tabellenzahl 2,1 Prozent mit "ja" antworten. Korrekt ist: Die Tabelle n...

--- Result 2 ---
Score: 0.5433 (similarity: 0.4567)
Source: chunking.pdf | Page: 1
Text: RAG Chunking Demo
2
1. Ausgangslage und Ziel
Im ersten Quartal lag die Fehlkommissionierungsquote im Lager Nord bei 3,8 Prozent. Besonders
haeufig wurden aehnlich verpackte Artikel verwechselt. Das Pi...

--- Result 3 ---
Score: 0.6441 (similarity: 0.3559)
Source: chunking.pdf | Page: 2
Text: Gesamtprogramm unterscheiden.
Einschraenkung: Die Aussage, dass die Barcode-Pruefung den groessten Effekt hatte, gilt nur
fuer dieses Lager und diesen achtwoechigen Zeitraum. Eine Uebertragung auf and...

--- Result 4 ---
Score: 0.6897 (similarity: 0.3103)
Source: chunking.pdf | Page: 1
Text: Die Analyse identifizierte d

# Metadata Pre-filtering

In [4]:
# Only search chunks from a specific document category
results_filtered = vectorstore.similarity_search(
    query,
    k=4,
    filter={"document_category": "company-policy"}
)

print(f"Results without filter: 4")
print(f"Results with category filter: {len(results_filtered)}")

for doc in results_filtered:
    print(f"\nSource: {doc.metadata.get('source')} | Category: {doc.metadata.get('document_category')}")
    print(doc.page_content[:200])

Results without filter: 4
Results with category filter: 4

Source: chunking.pdf | Category: company-policy
Testfall fuer Retrieval
Frage: "Hat die Barcode-Pruefung das Ziel allein erreicht?" Eine oberflaechliche Suche
koennte wegen der Tabellenzahl 2,1 Prozent mit "ja" antworten. Korrekt ist: Die Tabelle n

Source: chunking.pdf | Category: company-policy
RAG Chunking Demo
2
1. Ausgangslage und Ziel
Im ersten Quartal lag die Fehlkommissionierungsquote im Lager Nord bei 3,8 Prozent. Besonders
haeufig wurden aehnlich verpackte Artikel verwechselt. Das Pi

Source: chunking.pdf | Category: company-policy
Gesamtprogramm unterscheiden.
Einschraenkung: Die Aussage, dass die Barcode-Pruefung den groessten Effekt hatte, gilt nur
fuer dieses Lager und diesen achtwoechigen Zeitraum. Eine Uebertragung auf and

Source: chunking.pdf | Category: company-policy
Die Analyse identifizierte drei Hauptursachen: uneindeutige Stellplatzkennzeichnung, zu spaete
Plausibilitaetspruefung und unzureichende Rueckm

# 6. Build a Proper RAG System Prompt

In [5]:
from langchain_core.prompts import ChatPromptTemplate

RAG_SYSTEM_PROMPT = """You are a helpful assistant that answers questions based strictly on the provided context.

Rules:
- Only use information from the provided context to answer the question.
- If the context does not contain enough information to answer, say "I don't have enough information in my knowledge base to answer this."
- Always cite your sources: include the file name and page number for each claim you make.
- Be concise. Do not pad the answer with unnecessary text.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human", "{question}")
])

print("Prompt template ready.")
print(f"\nSystem message preview:\n{RAG_SYSTEM_PROMPT[:300]}...")

Prompt template ready.

System message preview:
You are a helpful assistant that answers questions based strictly on the provided context.

Rules:
- Only use information from the provided context to answer the question.
- If the context does not contain enough information to answer, say "I don't have enough information in my knowledge base to ans...


# Wie werden die Variablen im Prompt ersetzt?

`{context}` und `{question}` im `RAG_SYSTEM_PROMPT` sind **Platzhalter**. Der String ist bewusst **kein f-String** (kein `f"..."`): Beim Definieren gibt es ja noch keinen Kontext und keine Frage. Die Klammern bleiben deshalb erst mal stehen und werden erst später ausgefüllt, bei jeder Frage neu.

1. `ChatPromptTemplate.from_messages(...)` sucht in allen Messages nach `{name}` und merkt sich die Namen: `prompt.input_variables` → `['context', 'question']`.
2. `prompt.invoke({"context": ..., "question": ...})` ersetzt jeden Platzhalter durch den Wert mit **demselben Schlüssel** im Dict. Heraus kommt ein `ChatPromptValue` mit den fertig ausgefüllten Messages.
3. In der `rag_chain` baut der erste Schritt `{"context": retriever | format_docs, "question": RunnablePassthrough()}` genau dieses Dict. Darum müssen die Schlüssel dort exakt so heißen wie die Platzhalter.

Regeln:
- Fehlt ein Schlüssel, gibt es einen `KeyError` mit dem Namen des fehlenden Platzhalters. Zusätzliche Schlüssel werden einfach ignoriert.
- Soll im Prompt eine echte geschweifte Klammer stehen (z.B. ein JSON-Beispiel), muss man sie verdoppeln: `{{"antwort": "..."}}` wird zu `{"antwort": "..."}` und ist kein Platzhalter.

In [6]:
# Welche Platzhalter hat das Template gefunden?
print("input_variables:", prompt.input_variables)

# Platzhalter mit Testwerten fuellen - kein Retriever, kein LLM noetig.
# Jeder {name} im Template wird durch den Wert mit demselben Schluessel ersetzt.
demo = prompt.invoke({
    "context": "[Source: demo.pdf, page 1]\nDas Ziel war, die Fehlerquote zu senken.",
    "question": "Was war das Ziel?",
})
for msg in demo.to_messages():
    print(f"\n========== [{msg.type}] ==========\n{msg.content}")

# Fehlt ein Schluessel, sagt LangChain genau, welcher Platzhalter leer bleibt:
try:
    prompt.invoke({"question": "Was war das Ziel?"})
except KeyError as e:
    print("\nKeyError:", e.args[0].split("Note:")[0].strip())

input_variables: ['context', 'question']

========== [system] ==========
You are a helpful assistant that answers questions based strictly on the provided context.

Rules:
- Only use information from the provided context to answer the question.
- If the context does not contain enough information to answer, say "I don't have enough information in my knowledge base to answer this."
- Always cite your sources: include the file name and page number for each claim you make.
- Be concise. Do not pad the answer with unnecessary text.

Context:
[Source: demo.pdf, page 1]
Das Ziel war, die Fehlerquote zu senken.


========== [human] ==========
Was war das Ziel?

KeyError: Input to ChatPromptTemplate is missing variables {'context'}.  Expected: ['context', 'question'] Received: ['question']


# Assemble the Full RAG Chain

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Das LLM, das am Ende die Antwort formuliert. temperature=0 -> deterministisch,
# fuer faktenbasierte RAG-Antworten gewollt.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def format_docs(docs):
    """Formatiert die vom Retriever gelieferten Documents zu einem Kontext-String.

    Args:
        docs: Liste von Document-Objekten aus der similarity_search. Jedes hat
            page_content (den Chunk-Text) und metadata (u.a. source und page).

    Returns:
        Einen einzigen String fuer den {context}-Platzhalter im Prompt:
        jeder Chunk beginnt mit seiner Quellenangabe "[Source: <datei>, page <n>]"
        (damit das LLM zitieren kann), die Chunks sind durch "---" getrennt.
    """
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        parts.append(f"[Source: {source}, page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


# Verpackt den Vector Store als Runnable: String rein -> similarity_search
# mit k=4 -> Liste von 4 Documents raus.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# LCEL-Kette (LangChain Expression Language): `|` reicht die Ausgabe eines
# Schritts als Eingabe an den naechsten weiter, wie eine Unix-Pipe.
#
# rag_chain.invoke("Frage") laeuft so durch:
#
# 1. Das Dict ist ein RunnableParallel: BEIDE Zweige bekommen dieselbe Frage.
#      "context":  Frage -> retriever (4 Chunks) -> format_docs (ein String)
#      "question": RunnablePassthrough() reicht die Frage unveraendert durch
#    Ergebnis: {"context": "<4 Chunks als Text>", "question": "<Frage>"}
# 2. | prompt: fuellt die Platzhalter {context} und {question} im Template
#    -> fertige Message-Liste (System + Human)
# 3. | llm: schickt die Messages an gpt-4o-mini -> AIMessage
# 4. | StrOutputParser(): zieht den reinen Text aus der AIMessage
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Was war das Hauptziel des Pilotversuchs?")
print(answer)

Das Hauptziel des Pilotversuchs war, die Fehlkommissionierungsquote im Lager Nord innerhalb von acht Wochen auf unter 2,5 Prozent zu senken, ohne die durchschnittliche Bearbeitungszeit pro Auftrag um mehr als zehn Sekunden zu erhöhen (chunking.pdf, Seite 1).


# Was sind Runnables? Jeden Baustein einzeln aufrufen

Ein **Runnable** ist in LangChain alles, was man mit `|` in eine Kette stecken kann. Alle Runnables haben **dieselbe Schnittstelle**:

- `.invoke(x)`: eine Eingabe rein, eine Ausgabe raus
- `.batch([x1, x2, ...])`: mehrere Eingaben auf einmal
- `.stream(x)`: Ausgabe stückweise, z.B. Token für Token

Weil alle Bausteine gleich bedient werden, kann `|` sie beliebig verbinden: Die Ausgabe des einen ist die Eingabe des nächsten. Die fertige Kette ist selbst wieder ein Runnable, deshalb funktioniert `rag_chain.invoke(...)` genauso wie `prompt.invoke(...)`.

| Baustein in `rag_chain` | Eingabe | Ausgabe |
|---|---|---|
| `retriever` | Frage (`str`) | Liste von `Document`s |
| `format_docs` | Liste von `Document`s | Kontext (`str`) |
| `{"context": ..., "question": ...}` | Frage (`str`) | `dict` |
| `prompt` | `dict` mit `context` + `question` | `ChatPromptValue` (Messages) |
| `llm` | Messages | `AIMessage` |
| `StrOutputParser()` | `AIMessage` | `str` |

**Achtung:** Runnables sind keine normalen Python-Funktionen. `prompt(...)` geht nicht, es heißt `prompt.invoke(...)`. Umgekehrt ist `format_docs` eine ganz normale Funktion: LCEL verpackt sie in der Kette automatisch in ein `RunnableLambda`, und das Dict automatisch in ein `RunnableParallel`.

Die nächsten zwei Zellen spielen `rag_chain.invoke(...)` von Hand nach: die erste bis zum fertig ausgefüllten Prompt (noch **kein** API-Call), die zweite schickt ihn ans LLM.

In [8]:
# Jeder Baustein der Kette laesst sich auch einzeln aufrufen. Hier spielen wir
# rag_chain.invoke(question) Schritt fuer Schritt von Hand nach - bis zum
# fertig ausgefuellten Prompt, noch OHNE LLM-Call.
#
# Zwei Arten von "aufrufbar":
# - format_docs ist eine normale Python-Funktion -> callable, Aufruf mit ()
# - retriever, prompt, llm, Parser sind Runnables -> NICHT callable,
#   Aufruf mit .invoke(). In der Kette verpackt LCEL format_docs automatisch
#   in ein RunnableLambda, damit es sich wie die anderen Bausteine verhaelt.
from langchain_core.runnables import RunnableLambda

question = "Was war das Hauptziel des Pilotversuchs?"
print("callable(format_docs):", callable(format_docs))
print("callable(prompt):     ", callable(prompt))

# 1a. Retriever: Frage -> Liste von Documents
docs = retriever.invoke(question)
print(f"\n1a. retriever.invoke(question) -> {len(docs)} Documents")

# 1b. format_docs: Documents -> ein Kontext-String
context = format_docs(docs)
print(f"1b. format_docs(docs)          -> str mit {len(context)} Zeichen")
# Als Runnable verpackt (so wie in der Kette) kommt dasselbe raus:
assert RunnableLambda(format_docs).invoke(docs) == context

# 2. Prompt: Dict mit den Platzhaltern -> fertige Messages (System + Human)
prompt_value = prompt.invoke({"context": context, "question": question})
print(f"2.  prompt.invoke({{...}})       -> {type(prompt_value).__name__}")

# Kurzform: die Kette einfach vor dem LLM abschneiden -> gleicher Prompt
prompt_chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt
assert prompt_chain.invoke(question) == prompt_value

# So sieht der Prompt aus, der gleich an das LLM geht:
for msg in prompt_value.to_messages():
    print(f"\n========== [{msg.type}] ==========\n{msg.content}")

callable(format_docs): True
callable(prompt):      False

1a. retriever.invoke(question) -> 4 Documents
1b. format_docs(docs)          -> str mit 1805 Zeichen
2.  prompt.invoke({...})       -> ChatPromptValue

========== [system] ==========
You are a helpful assistant that answers questions based strictly on the provided context.

Rules:
- Only use information from the provided context to answer the question.
- If the context does not contain enough information to answer, say "I don't have enough information in my knowledge base to answer this."
- Always cite your sources: include the file name and page number for each claim you make.
- Be concise. Do not pad the answer with unnecessary text.

Context:
[Source: chunking.pdf, page 4]
Testfall fuer Retrieval
Frage: "Hat die Barcode-Pruefung das Ziel allein erreicht?" Eine oberflaechliche Suche
koennte wegen der Tabellenzahl 2,1 Prozent mit "ja" antworten. Korrekt ist: Die Tabelle nennt
2,1 Prozent fuer diese Massnahme, die Zielerreichung

In [9]:
# 3. LLM: Messages -> AIMessage (erst hier passiert der API-Call)
ai_message = llm.invoke(prompt_value)
print(f"3. llm.invoke(prompt_value)   -> {type(ai_message).__name__}")
print(f"   Tokens: {ai_message.usage_metadata}")

# 4. Parser: AIMessage -> reiner Text
answer_manual = StrOutputParser().invoke(ai_message)
print(f"4. StrOutputParser().invoke() -> str\n")
print(answer_manual)

3. llm.invoke(prompt_value)   -> AIMessage
   Tokens: {'input_tokens': 583, 'output_tokens': 59, 'total_tokens': 642, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
4. StrOutputParser().invoke() -> str

Das Hauptziel des Pilotversuchs war, die Fehlkommissionierungsquote im Lager Nord innerhalb von acht Wochen auf unter 2,5 Prozent zu senken, ohne die durchschnittliche Bearbeitungszeit pro Auftrag um mehr als zehn Sekunden zu erhöhen (chunking.pdf, Seite 1).


# Ask a Question the Store Can’t Answer

In [10]:
unanswerable = "What is the weather in Berlin today?"
answer_fail = rag_chain.invoke(unanswerable)
print(answer_fail)

I don't have enough information in my knowledge base to answer this.


# Tune k — See the Trade-off

In [11]:
for k in [1, 3, 6]:
    retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
    chain_k = (
        {"context": retriever_k | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    answer_k = chain_k.invoke("Was war das Hauptziel des Pilotversuchs?")
    print(f"\n=== k={k} ===")
    print(answer_k[:400])


=== k=1 ===
Ich habe nicht genug Informationen in meinem Wissensbestand, um diese Frage zu beantworten.

=== k=3 ===
Das Hauptziel des Pilotversuchs war, die Fehlkommissionierungsquote im Lager Nord innerhalb von acht Wochen auf unter 2,5 Prozent zu senken, ohne die durchschnittliche Bearbeitungszeit pro Auftrag um mehr als zehn Sekunden zu erhöhen (chunking.pdf, Seite 1).

=== k=6 ===
Das Hauptziel des Pilotversuchs war, die Fehlkommissionierungsquote im Lager Nord innerhalb von acht Wochen auf unter 2,5 Prozent zu senken, ohne die durchschnittliche Bearbeitungszeit pro Auftrag um mehr als zehn Sekunden zu erhöhen (chunking.pdf, Seite 1).


# Show Retrieved Sources Separately

In [12]:
def answer_with_sources(question, k=4):
    """Beantwortet eine Frage per RAG und listet die verwendeten Quellen auf.

    Args:
        question: Die Nutzerfrage als String.
        k: Anzahl der Chunks, die fuer die Quellen-Anzeige geholt werden.

    Hinweis: Die Chunks werden hier ein zweites Mal geholt, nur um sie anzeigen
    zu koennen — rag_chain macht intern sein eigenes Retrieval mit fixem k=4.
    Bei k != 4 koennen Anzeige und tatsaechlich genutzter Kontext also abweichen.
    """
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)
    answer = rag_chain.invoke(question)

    print("ANSWER:")
    print(answer)
    print("\nSOURCES USED:")
    for doc in docs:
        print(f"  - {doc.metadata.get('source')} (page {doc.metadata.get('page', '?')})")

answer_with_sources( "Was war das Hauptziel des Pilotversuchs?")

ANSWER:
Das Hauptziel des Pilotversuchs war, die Fehlkommissionierungsquote im Lager Nord innerhalb von acht Wochen auf unter 2,5 Prozent zu senken, ohne die durchschnittliche Bearbeitungszeit pro Auftrag um mehr als zehn Sekunden zu erhöhen (chunking.pdf, Seite 1).

SOURCES USED:
  - chunking.pdf (page 4)
  - chunking.pdf (page 1)
  - chunking.pdf (page 2)
  - chunking.pdf (page 1)
